# 35. 新しい足を受ける仮想売買基盤
出典：FX (3).ipynb、元セルindex [84, 85, 86]。保存出力は results/imported_fx3/。
研究履歴の原本です。Notebookの変数・価格CSV・学習済みファイルに依存します。
失敗した試行も保管しています。一括実行やAPI接続を開始する入口ではありません。
元コード内の指示・自動判定名は資料として保存しています。独立した検証済みの結論とは区別してください。


## 元セルindex 84
構文状態：valid


In [ ]:
# ============================================================
# FORWARD PAPER TRADING RUNNER — ONE CELL
#
# Frozen Champion: BASE_PLUS_REGIME
#
# このセルの役割
# ------------------------------------------------------------
# 1. Frozen Championを変更しない
# 2. PRODUCTION_CANONICAL_HISTORYを必須とする
# 3. 新しい確定15分足だけを受け付ける
# 4. 既存のHardened Live Inference Engineを再利用
# 5. Signal -> Entry 15分
# 6. Entry -> Exit 30分
# 7. Entry = OPEN_AT_TIME
# 8. Exit = PREVIOUS_BAR_CLOSE
# 9. Cost = SIZE_SCALED_COST
# 10. Position / Equity / Pending Orderを保存
# 11. Signal / Trade / EventをCSV保存
# 12. 重複・Overlap・履歴改変・異常OHLCなら停止
#
# PAPER ONLY
# 実際のブローカー注文は絶対に送信しません。
# ============================================================

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

import pandas as pd
import numpy as np


# ============================================================
# CONFIG
# ============================================================

STARTING_EQUITY = 1_000_000.0

# すでに検証済みのFrozen Cost
FROZEN_BASE_COST = 4e-05

BAR_MINUTES = 15
HOLD_MINUTES = 30

# USDJPY 15分足として明らかに異常なデータを止める安全装置
MAX_ABS_15M_MOVE = 0.03

ALLOWED_SIDES = {"BUY", "SELL"}


# ============================================================
# BASIC HELPERS
# ============================================================

def _utc(x):
    return pd.to_datetime(
        x,
        utc=True,
        errors="coerce"
    )


def _iso(x):
    x = _utc(x)

    if pd.isna(x):
        return None

    return x.isoformat()


def _float(x, default=np.nan):

    try:

        x = float(x)

        if np.isfinite(x):
            return x

    except Exception:
        pass

    return default


def _read_json(path, default):

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            return json.load(f)

    except Exception:

        return default


def _write_json(path, obj):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    tmp = Path(
        str(path) + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
            default=str
        )

    tmp.replace(path)


def _append_csv(path, rows):

    if len(rows) == 0:
        return

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df = pd.DataFrame(rows)

    exists = (
        path.exists()
        and path.stat().st_size > 0
    )

    df.to_csv(
        path,
        mode="a",
        header=not exists,
        index=False
    )


def _read_csv_safe(path):

    path = Path(path)

    if (
        not path.exists()
        or path.stat().st_size == 0
    ):

        return pd.DataFrame(
            columns=[
                "timestamp",
                "open",
                "high",
                "low",
                "close"
            ]
        )

    try:

        return pd.read_csv(path)

    except pd.errors.EmptyDataError:

        return pd.DataFrame(
            columns=[
                "timestamp",
                "open",
                "high",
                "low",
                "close"
            ]
        )


# ============================================================
# OHLC NORMALIZATION
# ============================================================

def _normalize_bars(df):

    x = df.copy()

    if "timestamp" in x.columns:

        index = pd.to_datetime(
            x["timestamp"],
            utc=True,
            errors="coerce"
        )

        x = x.drop(
            columns=["timestamp"]
        )

        x.index = index

    else:

        x.index = pd.to_datetime(
            x.index,
            utc=True,
            errors="coerce"
        )

    x.index.name = "timestamp"

    x = x[
        ~x.index.isna()
    ].copy()

    x = x.sort_index()

    for col in [
        "open",
        "high",
        "low",
        "close"
    ]:

        if col not in x.columns:

            raise ValueError(
                f"Missing OHLC column: {col}"
            )

        x[col] = pd.to_numeric(
            x[col],
            errors="coerce"
        )

    return x


def _valid_ohlc(row):

    try:

        o = float(row["open"])
        h = float(row["high"])
        l = float(row["low"])
        c = float(row["close"])

    except Exception:

        return False

    if not np.all(
        np.isfinite(
            [o, h, l, c]
        )
    ):

        return False

    if min(o, h, l, c) <= 0:

        return False

    if h < max(o, c):

        return False

    if l > min(o, c):

        return False

    if h < l:

        return False

    return True


def _exact_15m(index):

    if len(index) == 0:
        return True

    idx = pd.DatetimeIndex(index)

    return bool(

        np.all(
            np.isin(
                idx.minute,
                [0, 15, 30, 45]
            )
        )

        and np.all(
            idx.second == 0
        )

        and np.all(
            idx.microsecond == 0
        )

    )


# ============================================================
# FROZEN HISTORY HASH
# ============================================================

def _fingerprint(df):

    x = df[
        [
            "open",
            "high",
            "low",
            "close"
        ]
    ].copy()

    x = x.sort_index()

    hashed = pd.util.hash_pandas_object(
        x,
        index=True
    )

    return hashlib.sha256(
        hashed.values.tobytes()
    ).hexdigest()


# ============================================================
# FIND FROZEN CHAMPION DIRECTORY
# ============================================================

def _find_champion_directory():

    root = Path(
        "production_champion"
    )

    if not root.exists():
        return None

    candidates = []

    for path in root.iterdir():

        if not path.is_dir():
            continue

        if "base_plus_regime" in path.name.lower():

            candidates.append(path)

    if len(candidates) == 0:
        return None

    # 今まで使っているVersionを最優先
    exact = [

        p

        for p in candidates

        if "20260909" in p.name

    ]

    if len(exact) > 0:

        candidates = exact

    return sorted(
        candidates,
        key=lambda p: p.name
    )[-1]


# ============================================================
# LIVE ENGINE OUTPUT NORMALIZER
# ============================================================

def _normalize_signal(
    result,
    fallback_time
):

    if isinstance(
        result,
        pd.Series
    ):

        result = result.to_dict()

    elif not isinstance(
        result,
        dict
    ):

        if hasattr(
            result,
            "__dict__"
        ):

            result = dict(
                vars(result)
            )

        else:

            return None

    lower = {

        str(k).lower(): v

        for k, v
        in result.items()

    }

    def get_value(
        *names,
        default=None
    ):

        for name in names:

            if name.lower() in lower:

                return lower[
                    name.lower()
                ]

        return default

    action = str(
        get_value(
            "action",
            "decision",
            default=""
        )
    ).upper().strip()

    side = str(
        get_value(
            "side",
            "proposed_side",
            "trade_side",
            default=""
        )
    ).upper().strip()

    if action in ALLOWED_SIDES:

        side = action

    elif action in {
        "TRADE",
        "ENTER",
        "ENTRY",
        "OPEN"
    }:

        if side not in ALLOWED_SIDES:

            return None

        action = side

    elif (
        action == "NO_TRADE"
        or "NO_TRADE" in action
        or action in {
            "FLAT",
            "HOLD",
            "SKIP",
            "NONE"
        }
    ):

        action = "NO_TRADE"

    else:

        return None

    signal_time = _utc(
        get_value(
            "signal_time",
            "_diag_signal_time",
            "bar_time",
            "timestamp",
            default=fallback_time
        )
    )

    if pd.isna(signal_time):

        return None

    position_size = _float(
        get_value(
            "position_size",
            "size",
            default=0
        ),
        0
    )

    if action == "NO_TRADE":

        position_size = 0.0

    return {

        "signal_time":
            signal_time,

        "action":
            action,

        "side":
            side if side in ALLOWED_SIDES else None,

        "raw_p_up":
            _float(
                get_value(
                    "raw_p_up",
                    "p_up",
                    "raw_probability"
                )
            ),

        "calibrated_p_up":
            _float(
                get_value(
                    "calibrated_p_up",
                    "calibrated_probability",
                    "cal_p_up"
                )
            ),

        "confidence":
            _float(
                get_value(
                    "confidence"
                )
            ),

        "threshold":
            _float(
                get_value(
                    "threshold"
                )
            ),

        "threshold_allowed":
            bool(
                get_value(
                    "threshold_allowed",
                    default=(
                        action != "NO_TRADE"
                    )
                )
            ),

        "session":
            get_value(
                "session",
                default=None
            ),

        "session_allowed":
            bool(
                get_value(
                    "session_allowed",
                    default=True
                )
            ),

        "position_size":
            position_size,

        "reason":
            str(
                get_value(
                    "reason",
                    default=""
                )
            )

    }


# ============================================================
# FIND EXISTING HARDENED LIVE INFERENCE FUNCTION
# ============================================================

def _find_live_engine(
    frozen_history
):

    G = globals()

    preferred = [

        "HARDENED_LIVE_INFERENCE",

        "hardened_live_inference",

        "run_hardened_live_inference",

        "LIVE_INFERENCE_ENGINE",

        "live_inference_engine",

        "run_live_inference",

        "live_inference",

        "frozen_live_inference",

        "production_live_inference"

    ]

    candidates = []

    seen = set()

    for name in preferred:

        if name in G:

            obj = G[name]

            if id(obj) not in seen:

                candidates.append(
                    (name, obj)
                )

                seen.add(
                    id(obj)
                )

    # 既存Notebook内から補助検索
    for name, obj in list(
        G.items()
    ):

        lower = str(
            name
        ).lower()

        if id(obj) in seen:
            continue

        strong_name = (

            (
                "live" in lower
                and "infer" in lower
            )

            or

            (
                "harden" in lower
                and "infer" in lower
            )

            or

            (
                "frozen" in lower
                and "infer" in lower
            )

        )

        if strong_name:

            candidates.append(
                (name, obj)
            )

            seen.add(
                id(obj)
            )

    attempts = []

    for name, obj in candidates:

        functions = []

        if callable(obj):

            functions.append(
                (name, obj)
            )

        for method_name in [

            "infer",
            "predict_one",
            "run",
            "process",
            "process_bar",
            "signal"

        ]:

            if (

                hasattr(
                    obj,
                    method_name
                )

                and callable(
                    getattr(
                        obj,
                        method_name
                    )
                )

            ):

                functions.append(

                    (

                        f"{name}.{method_name}",

                        getattr(
                            obj,
                            method_name
                        )

                    )

                )

        for fn_name, fn in functions:

            attempts.append(
                (
                    fn_name,
                    fn
                )
            )

    for name, fn in attempts:

        call_patterns = [

            (
                (frozen_history,),
                {}
            ),

            (
                (),
                {
                    "history":
                        frozen_history
                }
            ),

            (
                (),
                {
                    "canonical_history":
                        frozen_history
                }
            ),

            (
                (),
                {
                    "bars":
                        frozen_history
                }
            ),

            (
                (),
                {
                    "df":
                        frozen_history
                }
            )

        ]

        for args, kwargs in call_patterns:

            try:

                result = fn(
                    *args,
                    **kwargs
                )

                normalized = _normalize_signal(
                    result,
                    frozen_history.index[-1]
                )

                if normalized is None:
                    continue

                if (
                    normalized[
                        "signal_time"
                    ]
                    != frozen_history.index[-1]
                ):

                    continue

                return (
                    name,
                    fn,
                    normalized
                )

            except Exception:

                continue

    return (
        None,
        None,
        None
    )


# ============================================================
# BUILD FORWARD PAPER SYSTEM
# ============================================================

def _build_forward_paper_system():

    G = globals()

    print(
        "=" * 90
    )

    print(
        "FORWARD PAPER TRADING SYSTEM BUILD"
    )

    print(
        "=" * 90
    )

    # ========================================================
    # PRIOR VALIDATION CHECK
    # ========================================================

    prior_flags = [

        "LIVE_ENGINE_HARDENING_PASSED",

        "FINAL_2026_PAPER_REPLAY_PASSED"

    ]

    for flag in prior_flags:

        if (
            flag in G
            and G[flag] is False
        ):

            print(
                f"STOP: {flag} = False"
            )

            return {
                "ready": False,
                "reason": flag
            }

    # ========================================================
    # CHAMPION
    # ========================================================

    champion_dir = (
        _find_champion_directory()
    )

    if champion_dir is None:

        print(
            "STOP:"
        )

        print(
            "Frozen BASE_PLUS_REGIME directory not found."
        )

        return {
            "ready": False,
            "reason":
                "CHAMPION_DIRECTORY_MISSING"
        }

    version = champion_dir.name

    print(
        "Champion:",
        "BASE_PLUS_REGIME"
    )

    print(
        "Version:",
        version
    )

    # ========================================================
    # CANONICAL HISTORY
    # ========================================================

    if (

        "PRODUCTION_CANONICAL_HISTORY"
        not in G

        or not isinstance(
            G[
                "PRODUCTION_CANONICAL_HISTORY"
            ],
            pd.DataFrame
        )

    ):

        print(
            "STOP:"
        )

        print(
            "PRODUCTION_CANONICAL_HISTORY is missing."
        )

        print(
            "Raw-data fallback is intentionally prohibited."
        )

        return {
            "ready": False,
            "reason":
                "CANONICAL_HISTORY_MISSING"
        }

    try:

        frozen = _normalize_bars(

            G[
                "PRODUCTION_CANONICAL_HISTORY"
            ]

        )

    except Exception as e:

        print(
            "STOP:"
        )

        print(
            "Canonical history validation failed."
        )

        print(
            str(e)
        )

        return {
            "ready": False,
            "reason":
                "CANONICAL_HISTORY_INVALID"
        }

    if (
        frozen.index
        .duplicated()
        .any()
    ):

        print(
            "STOP: duplicate canonical timestamps."
        )

        return {
            "ready": False,
            "reason":
                "CANONICAL_DUPLICATES"
        }

    if not _exact_15m(
        frozen.index
    ):

        print(
            "STOP: canonical history is not exact 15m."
        )

        return {
            "ready": False,
            "reason":
                "CANONICAL_GRID_INVALID"
        }

    invalid_ohlc = [

        ts

        for ts, row
        in frozen.iterrows()

        if not _valid_ohlc(
            row
        )

    ]

    if len(
        invalid_ohlc
    ) > 0:

        print(
            "STOP:"
        )

        print(
            "Invalid frozen OHLC:",
            invalid_ohlc[:5]
        )

        return {
            "ready": False,
            "reason":
                "CANONICAL_OHLC_INVALID"
        }

    frozen_rows = len(
        frozen
    )

    frozen_latest = (
        frozen.index[-1]
    )

    frozen_hash = (
        _fingerprint(
            frozen
        )
    )

    print(
        "Frozen rows:",
        f"{frozen_rows:,}"
    )

    print(
        "Frozen latest:",
        frozen_latest
    )

    # ========================================================
    # EXISTING HARDENED LIVE ENGINE
    # ========================================================

    engine_name, engine_fn, smoke = (
        _find_live_engine(
            frozen
        )
    )

    if engine_fn is None:

        print(
            "STOP:"
        )

        print(
            "Validated Hardened Live Inference callable not found."
        )

        print(
            "No fallback model was used."
        )

        return {
            "ready": False,
            "reason":
                "LIVE_ENGINE_NOT_FOUND"
        }

    print(
        "Live engine:",
        engine_name
    )

    print(
        "Smoke signal:",
        smoke[
            "signal_time"
        ]
    )

    print(
        "Smoke action:",
        smoke[
            "action"
        ]
    )

    # ========================================================
    # PAPER DIRECTORIES
    # ========================================================

    paper_dir = (

        champion_dir

        / "runtime"

        / "paper_forward"

    )

    paper_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    feed_path = (

        paper_dir

        / "usdjpy_15m_forward_feed.csv"

    )

    state_path = (

        paper_dir

        / "paper_state.json"

    )

    signals_path = (

        paper_dir

        / "paper_signals.csv"

    )

    trades_path = (

        paper_dir

        / "paper_trades.csv"

    )

    events_path = (

        paper_dir

        / "paper_events.csv"

    )

    contract_path = (

        paper_dir

        / "forward_paper_contract.json"

    )

    if not feed_path.exists():

        pd.DataFrame(

            columns=[

                "timestamp",

                "open",

                "high",

                "low",

                "close"

            ]

        ).to_csv(
            feed_path,
            index=False
        )

    # ========================================================
    # PAPER CONTRACT
    # ========================================================

    contract = {

        "paper_only":
            True,

        "champion":
            "BASE_PLUS_REGIME",

        "version":
            version,

        "frozen_rows":
            frozen_rows,

        "frozen_latest":
            _iso(
                frozen_latest
            ),

        "frozen_history_hash":
            frozen_hash,

        "feature_count":
            41,

        "bar_minutes":
            15,

        "signal_to_entry_minutes":
            15,

        "entry_to_exit_minutes":
            30,

        "entry_semantics":
            "OPEN_AT_TIME",

        "exit_semantics":
            "PREVIOUS_BAR_CLOSE",

        "return_formula":
            "SIGNED_SIMPLE_RETURN",

        "cost_mode":
            "SIZE_SCALED_COST",

        "base_cost":
            FROZEN_BASE_COST,

        "net_return_formula":
            "position_size * (gross_return - base_cost)",

        "overlap":
            "PROHIBITED",

        "real_orders":
            False,

        "inference_callable":
            engine_name

    }

    _write_json(
        contract_path,
        contract
    )

    # ========================================================
    # DEFAULT STATE
    # ========================================================

    def default_state():

        return {

            "paper_only":
                True,

            "champion":
                "BASE_PLUS_REGIME",

            "version":
                version,

            "frozen_history_hash":
                frozen_hash,

            "frozen_latest":
                _iso(
                    frozen_latest
                ),

            "last_processed_bar":
                _iso(
                    frozen_latest
                ),

            "equity":
                STARTING_EQUITY,

            "pending_order":
                None,

            "open_position":
                None,

            "halted":
                False,

            "halt_reason":
                None,

            "processed_forward_bars":
                0,

            "signals_seen":
                0,

            "trades_closed":
                0,

            "updated_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat()

        }

    if state_path.exists():

        state = _read_json(
            state_path,
            default_state()
        )

        if (
            state.get(
                "version"
            )
            != version
        ):

            print(
                "STOP:"
            )

            print(
                "Existing state belongs to another Champion."
            )

            return {
                "ready": False,
                "reason":
                    "STATE_VERSION_MISMATCH"
            }

        if (
            state.get(
                "frozen_history_hash"
            )
            != frozen_hash
        ):

            print(
                "STOP:"
            )

            print(
                "Frozen historical prefix has changed."
            )

            return {
                "ready": False,
                "reason":
                    "FROZEN_HISTORY_CHANGED"
            }

    else:

        state = (
            default_state()
        )

        _write_json(
            state_path,
            state
        )

    # ========================================================
    # LIVE INFERENCE ADAPTER
    # ========================================================

    def infer_full_history(
        history,
        existing_position=False
    ):

        attempts = [

            (
                (history,),
                {}
            ),

            (
                (),
                {
                    "history":
                        history
                }
            ),

            (
                (),
                {
                    "canonical_history":
                        history
                }
            ),

            (
                (),
                {
                    "bars":
                        history
                }
            ),

            (
                (),
                {
                    "df":
                        history
                }
            ),

            (
                (history,),
                {
                    "existing_position":
                        existing_position
                }
            ),

            (
                (),
                {
                    "history":
                        history,

                    "existing_position":
                        existing_position
                }
            ),

            (
                (),
                {
                    "canonical_history":
                        history,

                    "existing_position":
                        existing_position
                }
            )

        ]

        last_error = None

        for args, kwargs in attempts:

            try:

                result = engine_fn(
                    *args,
                    **kwargs
                )

                signal = _normalize_signal(
                    result,
                    history.index[-1]
                )

                if signal is not None:

                    return signal

            except Exception as e:

                last_error = e

        raise RuntimeError(

            "Frozen live inference failed. "
            f"Last error: {repr(last_error)}"

        )

    # ========================================================
    # LOAD FORWARD HISTORY
    # ========================================================

    def load_forward_history():

        feed_raw = (
            _read_csv_safe(
                feed_path
            )
        )

        if feed_raw.empty:

            return (
                frozen.copy(),
                pd.DataFrame(
                    columns=frozen.columns
                )
            )

        required = {

            "timestamp",

            "open",

            "high",

            "low",

            "close"

        }

        missing = (
            required
            - set(
                feed_raw.columns
            )
        )

        if len(
            missing
        ) > 0:

            raise ValueError(

                f"Feed missing columns: {sorted(missing)}"

            )

        feed = _normalize_bars(

            feed_raw[
                [
                    "timestamp",
                    "open",
                    "high",
                    "low",
                    "close"
                ]
            ]

        )

        if (
            feed.index
            .duplicated()
            .any()
        ):

            raise ValueError(

                "Duplicate timestamps in forward feed."

            )

        if not _exact_15m(
            feed.index
        ):

            raise ValueError(

                "Off-grid timestamps in forward feed."

            )

        invalid = [

            ts

            for ts, row
            in feed.iterrows()

            if not _valid_ohlc(
                row
            )

        ]

        if len(
            invalid
        ) > 0:

            raise ValueError(

                f"Invalid OHLC rows: {invalid[:5]}"

            )

        # ====================================================
        # Frozen history mutation protection
        # ====================================================

        old_part = feed.loc[
            feed.index
            <= frozen_latest
        ]

        for ts, row in old_part.iterrows():

            if ts not in frozen.index:

                raise ValueError(

                    f"Feed inserted timestamp inside frozen history: {ts}"

                )

            old_prices = frozen.loc[
                ts,
                [
                    "open",
                    "high",
                    "low",
                    "close"
                ]
            ].astype(float).values

            new_prices = row[
                [
                    "open",
                    "high",
                    "low",
                    "close"
                ]
            ].astype(float).values

            if not np.allclose(

                old_prices,

                new_prices,

                rtol=0,

                atol=1e-12

            ):

                raise ValueError(

                    f"Frozen history conflict at {ts}"

                )

        forward = feed.loc[
            feed.index
            > frozen_latest
        ].copy()

        # ====================================================
        # Extreme-price safety
        # ====================================================

        check = pd.concat(

            [
                frozen.tail(1),
                forward
            ]

        ).sort_index()

        if len(
            check
        ) > 1:

            previous_close = (
                check[
                    "close"
                ].shift(1)
            )

            gap_move = (

                check[
                    "open"
                ]
                / previous_close
                - 1

            ).abs()

            bar_move = (

                check[
                    "close"
                ]
                / check[
                    "open"
                ]
                - 1

            ).abs()

            abnormal = check.index[

                (
                    gap_move
                    > MAX_ABS_15M_MOVE
                )

                |

                (
                    bar_move
                    > MAX_ABS_15M_MOVE
                )

            ]

            abnormal = [

                ts

                for ts in abnormal

                if ts > frozen_latest

            ]

            if len(
                abnormal
            ) > 0:

                raise ValueError(

                    "Extreme price move detected: "
                    f"{abnormal[:5]}"

                )

        history = pd.concat(

            [
                frozen,
                forward
            ]

        ).sort_index()

        history = history[
            ~history.index.duplicated(
                keep="first"
            )
        ]

        return (
            history,
            forward
        )

    # ========================================================
    # STATE SAVE
    # ========================================================

    def save_state(st):

        st[
            "updated_at_utc"
        ] = datetime.now(
            timezone.utc
        ).isoformat()

        _write_json(
            state_path,
            st
        )

    # ========================================================
    # EVENTS
    # ========================================================

    def record_event(
        event_type,
        event_time,
        detail="",
        **kwargs
    ):

        row = {

            "recorded_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "event_time":
                _iso(
                    event_time
                ),

            "event_type":
                event_type,

            "detail":
                detail

        }

        row.update(
            kwargs
        )

        _append_csv(
            events_path,
            [row]
        )

    # ========================================================
    # HALT
    # ========================================================

    def halt(
        st,
        reason,
        when=None
    ):

        st[
            "halted"
        ] = True

        st[
            "halt_reason"
        ] = str(
            reason
        )

        save_state(
            st
        )

        record_event(

            "HALT",

            when
            if when is not None
            else datetime.now(
                timezone.utc
            ),

            str(
                reason
            )

        )

        print()

        print(
            "=" * 90
        )

        print(
            "PAPER ENGINE HALTED SAFELY"
        )

        print(
            "=" * 90
        )

        print(
            "Reason:",
            reason
        )

        print(
            "No further paper trade was generated."
        )

        return st

    # ========================================================
    # PUBLIC:
    # ADD ONE NEW CLOSED 15m BAR
    # ========================================================

    def paper_append_bar(
        timestamp,
        open_,
        high,
        low,
        close
    ):

        ts = _utc(
            timestamp
        )

        if pd.isna(ts):

            print(
                "REJECTED: invalid timestamp."
            )

            return False

        if not (

            ts.minute
            in [
                0,
                15,
                30,
                45
            ]

            and ts.second == 0

            and ts.microsecond == 0

        ):

            print(
                "REJECTED: timestamp is not exact 15m."
            )

            return False

        row = {

            "timestamp":
                ts.isoformat(),

            "open":
                _float(
                    open_
                ),

            "high":
                _float(
                    high
                ),

            "low":
                _float(
                    low
                ),

            "close":
                _float(
                    close
                )

        }

        if not _valid_ohlc(
            pd.Series(row)
        ):

            print(
                "REJECTED: invalid OHLC."
            )

            return False

        current = (
            _read_csv_safe(
                feed_path
            )
        )

        if not current.empty:

            existing_time = pd.to_datetime(

                current[
                    "timestamp"
                ],

                utc=True,

                errors="coerce"

            )

            same = (
                existing_time
                == ts
            )

            if same.any():

                old = (
                    current.loc[
                        same
                    ].iloc[-1]
                )

                old_price = np.array(

                    [
                        old["open"],
                        old["high"],
                        old["low"],
                        old["close"]
                    ],

                    dtype=float

                )

                new_price = np.array(

                    [
                        row["open"],
                        row["high"],
                        row["low"],
                        row["close"]
                    ],

                    dtype=float

                )

                if np.allclose(

                    old_price,

                    new_price,

                    rtol=0,

                    atol=1e-12

                ):

                    print(
                        "IGNORED: identical bar already exists."
                    )

                    return True

                print(
                    "REJECTED: conflicting timestamp already exists."
                )

                return False

        pd.DataFrame(
            [row]
        ).to_csv(

            feed_path,

            mode="a",

            header=(
                not feed_path.exists()
                or feed_path.stat().st_size == 0
            ),

            index=False

        )

        print(
            "APPENDED:",
            ts
        )

        return True

    # ========================================================
    # MAIN FORWARD PAPER RUNNER
    # ========================================================

    def run_forward_paper(
        max_bars=None
    ):

        st = _read_json(
            state_path,
            default_state()
        )

        if st.get(
            "halted",
            False
        ):

            print(
                "PAPER ENGINE IS HALTED."
            )

            print(
                "Reason:",
                st.get(
                    "halt_reason"
                )
            )

            return st

        if (

            st.get(
                "version"
            )
            != version

            or

            st.get(
                "frozen_history_hash"
            )
            != frozen_hash

        ):

            return halt(

                st,

                "Champion or frozen history changed."

            )

        try:

            history, forward = (
                load_forward_history()
            )

        except Exception as e:

            return halt(

                st,

                "Feed validation failed: "
                + str(e)

            )

        last_processed = _utc(

            st.get(
                "last_processed_bar"
            )

        )

        new_bars = forward.loc[

            forward.index
            > last_processed

        ].copy()

        if max_bars is not None:

            new_bars = new_bars.iloc[
                :int(max_bars)
            ]

        print(
            "=" * 90
        )

        print(
            "FORWARD PAPER RUN"
        )

        print(
            "=" * 90
        )

        print(
            "Champion:",
            version
        )

        print(
            "Last processed:",
            last_processed
        )

        print(
            "New closed bars:",
            len(new_bars)
        )

        print(
            "Equity:",
            f"{float(st['equity']):,.2f}"
        )

        print(
            "Pending:",
            st.get(
                "pending_order"
            )
        )

        print(
            "Position:",
            st.get(
                "open_position"
            )
        )

        if new_bars.empty:

            print()

            print(
                "READY — no new closed 15m bars."
            )

            return st

        signal_log = []

        trade_log = []

        for bar_time, bar in new_bars.iterrows():

            bar_close_time = (

                bar_time

                + pd.Timedelta(
                    minutes=15
                )

            )

            # =================================================
            # 1. PENDING ENTRY
            # =================================================

            pending = st.get(
                "pending_order"
            )

            if pending is not None:

                entry_time = _utc(

                    pending[
                        "entry_time"
                    ]

                )

                if entry_time < bar_time:

                    return halt(

                        st,

                        (
                            "Required entry bar was missed. "
                            f"entry={entry_time}, current={bar_time}"
                        ),

                        bar_time

                    )

                if entry_time == bar_time:

                    if (
                        st.get(
                            "open_position"
                        )
                        is not None
                    ):

                        return halt(

                            st,

                            "Overlap at entry.",

                            bar_time

                        )

                    entry_price = float(
                        bar[
                            "open"
                        ]
                    )

                    st[
                        "open_position"
                    ] = {

                        "signal_time":
                            pending[
                                "signal_time"
                            ],

                        "entry_time":
                            pending[
                                "entry_time"
                            ],

                        "exit_time":
                            pending[
                                "exit_time"
                            ],

                        "side":
                            pending[
                                "side"
                            ],

                        "position_size":
                            float(
                                pending[
                                    "position_size"
                                ]
                            ),

                        "entry_price":
                            entry_price,

                        "confidence":
                            pending.get(
                                "confidence"
                            ),

                        "raw_p_up":
                            pending.get(
                                "raw_p_up"
                            ),

                        "calibrated_p_up":
                            pending.get(
                                "calibrated_p_up"
                            )

                    }

                    st[
                        "pending_order"
                    ] = None

                    record_event(

                        "ENTRY",

                        entry_time,

                        side=st[
                            "open_position"
                        ][
                            "side"
                        ],

                        price=
                            entry_price,

                        position_size=
                            st[
                                "open_position"
                            ][
                                "position_size"
                            ]

                    )

            # =================================================
            # 2. EXIT
            # =================================================

            position = st.get(
                "open_position"
            )

            if position is not None:

                exit_time = _utc(

                    position[
                        "exit_time"
                    ]

                )

                if (
                    exit_time
                    < bar_close_time
                ):

                    return halt(

                        st,

                        (
                            "Required exit was missed. "
                            f"exit={exit_time}, "
                            f"current close={bar_close_time}"
                        ),

                        bar_time

                    )

                if (
                    exit_time
                    == bar_close_time
                ):

                    exit_price = float(
                        bar[
                            "close"
                        ]
                    )

                    entry_price = float(

                        position[
                            "entry_price"
                        ]

                    )

                    side = str(

                        position[
                            "side"
                        ]

                    ).upper()

                    size = float(

                        position[
                            "position_size"
                        ]

                    )

                    raw_price_return = (

                        exit_price
                        / entry_price
                        - 1.0

                    )

                    if side == "BUY":

                        gross_return = (
                            raw_price_return
                        )

                    else:

                        gross_return = (
                            -raw_price_return
                        )

                    # Frozen verified accounting
                    net_return = (

                        size

                        * (

                            gross_return

                            - FROZEN_BASE_COST

                        )

                    )

                    equity_before = float(

                        st[
                            "equity"
                        ]

                    )

                    equity_after = (

                        equity_before

                        * (
                            1.0
                            + net_return
                        )

                    )

                    trade = {

                        "signal_time":
                            position[
                                "signal_time"
                            ],

                        "entry_time":
                            position[
                                "entry_time"
                            ],

                        "exit_time":
                            position[
                                "exit_time"
                            ],

                        "side":
                            side,

                        "position_size":
                            size,

                        "entry_price":
                            entry_price,

                        "exit_price":
                            exit_price,

                        "gross_return":
                            gross_return,

                        "base_cost":
                            FROZEN_BASE_COST,

                        "net_return":
                            net_return,

                        "equity_before":
                            equity_before,

                        "equity_after":
                            equity_after,

                        "confidence":
                            position.get(
                                "confidence"
                            ),

                        "raw_p_up":
                            position.get(
                                "raw_p_up"
                            ),

                        "calibrated_p_up":
                            position.get(
                                "calibrated_p_up"
                            ),

                        "champion_version":
                            version

                    }

                    trade_log.append(
                        trade
                    )

                    st[
                        "equity"
                    ] = equity_after

                    st[
                        "trades_closed"
                    ] = (

                        int(
                            st.get(
                                "trades_closed",
                                0
                            )
                        )

                        + 1

                    )

                    st[
                        "open_position"
                    ] = None

                    record_event(

                        "EXIT",

                        exit_time,

                        side=
                            side,

                        price=
                            exit_price,

                        gross_return=
                            gross_return,

                        net_return=
                            net_return,

                        equity=
                            equity_after

                    )

            # =================================================
            # 3. FULL CANONICAL HISTORY INFERENCE
            # =================================================

            history_until_now = (
                history.loc[
                    :bar_time
                ].copy()
            )

            try:

                signal = (
                    infer_full_history(

                        history_until_now,

                        existing_position=(

                            st.get(
                                "open_position"
                            )
                            is not None

                            or

                            st.get(
                                "pending_order"
                            )
                            is not None

                        )

                    )
                )

            except Exception as e:

                return halt(

                    st,

                    "Live inference failed: "
                    + str(e),

                    bar_time

                )

            if (
                signal[
                    "signal_time"
                ]
                != bar_time
            ):

                return halt(

                    st,

                    (
                        "Signal timestamp mismatch. "
                        f"engine={signal['signal_time']} "
                        f"expected={bar_time}"
                    ),

                    bar_time

                )

            st[
                "signals_seen"
            ] = (

                int(
                    st.get(
                        "signals_seen",
                        0
                    )
                )

                + 1

            )

            signal_row = {

                "signal_time":
                    _iso(
                        bar_time
                    ),

                "bar_close_time":
                    _iso(
                        bar_close_time
                    ),

                "action":
                    signal[
                        "action"
                    ],

                "side":
                    signal[
                        "side"
                    ],

                "reason":
                    signal[
                        "reason"
                    ],

                "raw_p_up":
                    signal[
                        "raw_p_up"
                    ],

                "calibrated_p_up":
                    signal[
                        "calibrated_p_up"
                    ],

                "confidence":
                    signal[
                        "confidence"
                    ],

                "threshold":
                    signal[
                        "threshold"
                    ],

                "threshold_allowed":
                    signal[
                        "threshold_allowed"
                    ],

                "session":
                    signal[
                        "session"
                    ],

                "session_allowed":
                    signal[
                        "session_allowed"
                    ],

                "position_size":
                    signal[
                        "position_size"
                    ],

                "champion_version":
                    version

            }

            # =================================================
            # TRADE SIGNAL
            # =================================================

            if (
                signal[
                    "action"
                ]
                in ALLOWED_SIDES
            ):

                if (

                    st.get(
                        "open_position"
                    )
                    is not None

                    or

                    st.get(
                        "pending_order"
                    )
                    is not None

                ):

                    signal_row[
                        "executed_action"
                    ] = "NO_TRADE"

                    signal_row[
                        "execution_reason"
                    ] = "OVERLAP_PROHIBITED"

                else:

                    position_size = float(

                        signal[
                            "position_size"
                        ]

                    )

                    if (

                        not np.isfinite(
                            position_size
                        )

                        or position_size <= 0

                    ):

                        return halt(

                            st,

                            (
                                "Invalid frozen position size: "
                                f"{position_size}"
                            ),

                            bar_time

                        )

                    entry_time = (

                        bar_time

                        + pd.Timedelta(
                            minutes=15
                        )

                    )

                    exit_time = (

                        entry_time

                        + pd.Timedelta(
                            minutes=30
                        )

                    )

                    st[
                        "pending_order"
                    ] = {

                        "signal_time":
                            _iso(
                                bar_time
                            ),

                        "entry_time":
                            _iso(
                                entry_time
                            ),

                        "exit_time":
                            _iso(
                                exit_time
                            ),

                        "side":
                            signal[
                                "action"
                            ],

                        "position_size":
                            position_size,

                        "confidence":
                            signal[
                                "confidence"
                            ],

                        "raw_p_up":
                            signal[
                                "raw_p_up"
                            ],

                        "calibrated_p_up":
                            signal[
                                "calibrated_p_up"
                            ]

                    }

                    signal_row[
                        "executed_action"
                    ] = signal[
                        "action"
                    ]

                    signal_row[
                        "execution_reason"
                    ] = "PAPER_ENTRY_SCHEDULED"

                    record_event(

                        "SIGNAL_TRADE",

                        bar_close_time,

                        side=
                            signal[
                                "action"
                            ],

                        entry_time=
                            _iso(
                                entry_time
                            ),

                        exit_time=
                            _iso(
                                exit_time
                            ),

                        position_size=
                            position_size

                    )

            else:

                signal_row[
                    "executed_action"
                ] = "NO_TRADE"

                signal_row[
                    "execution_reason"
                ] = (

                    signal[
                        "reason"
                    ]

                    or

                    "FROZEN_ENGINE_NO_TRADE"

                )

            signal_log.append(
                signal_row
            )

            # =================================================
            # SAVE AFTER EVERY BAR
            # =================================================

            st[
                "last_processed_bar"
            ] = _iso(
                bar_time
            )

            st[
                "processed_forward_bars"
            ] = (

                int(
                    st.get(
                        "processed_forward_bars",
                        0
                    )
                )

                + 1

            )

            save_state(
                st
            )

        # ====================================================
        # SAVE LOGS
        # ====================================================

        _append_csv(
            signals_path,
            signal_log
        )

        _append_csv(
            trades_path,
            trade_log
        )

        print()

        print(
            "=" * 90
        )

        print(
            "FORWARD PAPER RUN COMPLETE"
        )

        print(
            "=" * 90
        )

        print(
            "Processed bars:",
            len(new_bars)
        )

        print(
            "Signals:",
            len(signal_log)
        )

        print(
            "Trades closed this run:",
            len(trade_log)
        )

        print(
            "Total closed trades:",
            st.get(
                "trades_closed",
                0
            )
        )

        print(
            "Current equity:",
            f"{float(st['equity']):,.2f}"
        )

        print(
            "Pending order:",
            st.get(
                "pending_order"
            )
        )

        print(
            "Open position:",
            st.get(
                "open_position"
            )
        )

        print(
            "Last processed:",
            st.get(
                "last_processed_bar"
            )
        )

        return st

    # ========================================================
    # STATUS
    # ========================================================

    def paper_status():

        st = _read_json(
            state_path,
            default_state()
        )

        print(
            "=" * 90
        )

        print(
            "FORWARD PAPER STATUS"
        )

        print(
            "=" * 90
        )

        print(
            json.dumps(
                st,
                ensure_ascii=False,
                indent=2
            )
        )

        print()

        print(
            "Feed:",
            feed_path
        )

        print(
            "Signals:",
            signals_path
        )

        print(
            "Trades:",
            trades_path
        )

        print(
            "Events:",
            events_path
        )

        return st

    # ========================================================
    # MANUAL HALT CLEAR
    # ========================================================

    def paper_clear_halt(
        confirm=False
    ):

        st = _read_json(
            state_path,
            default_state()
        )

        if not confirm:

            print(
                "HALT was NOT cleared."
            )

            print(
                "After diagnosing the cause, run:"
            )

            print(
                "paper_clear_halt(confirm=True)"
            )

            return st

        st[
            "halted"
        ] = False

        st[
            "halt_reason"
        ] = None

        save_state(
            st
        )

        record_event(

            "HALT_CLEARED",

            datetime.now(
                timezone.utc
            ),

            "Manual clear after diagnosis."

        )

        print(
            "HALT cleared."
        )

        print(
            "Trading history was NOT deleted."
        )

        return st

    # ========================================================
    # EXPOSE TO NOTEBOOK
    # ========================================================

    G[
        "paper_append_bar"
    ] = paper_append_bar

    G[
        "run_forward_paper"
    ] = run_forward_paper

    G[
        "paper_status"
    ] = paper_status

    G[
        "paper_clear_halt"
    ] = paper_clear_halt

    G[
        "FORWARD_PAPER_FEED_PATH"
    ] = feed_path

    G[
        "FORWARD_PAPER_STATE_PATH"
    ] = state_path

    G[
        "FORWARD_PAPER_SIGNALS_PATH"
    ] = signals_path

    G[
        "FORWARD_PAPER_TRADES_PATH"
    ] = trades_path

    G[
        "FORWARD_PAPER_EVENTS_PATH"
    ] = events_path

    G[
        "FORWARD_PAPER_CONTRACT_PATH"
    ] = contract_path

    G[
        "FORWARD_PAPER_READY"
    ] = True

    print()

    print(
        "=" * 90
    )

    print(
        "FORWARD PAPER SYSTEM BUILD PASSED"
    )

    print(
        "=" * 90
    )

    print(
        "Champion: BASE_PLUS_REGIME"
    )

    print(
        "Version:",
        version
    )

    print(
        "Feature count: 41"
    )

    print(
        "Frozen history protected: True"
    )

    print(
        "Overlap prohibited: True"
    )

    print(
        "Paper only: True"
    )

    print(
        "Real broker orders: False"
    )

    print(
        "State persistence: True"
    )

    print(
        "Signal log: True"
    )

    print(
        "Trade log: True"
    )

    print(
        "Fail-safe: True"
    )

    print()

    print(
        "FORWARD_PAPER_READY: True"
    )

    print()

    print(
        "Feed file:"
    )

    print(
        feed_path
    )

    print()

    print(
        "NEXT COMMAND:"
    )

    print(
        "run_forward_paper()"
    )

    return {

        "ready":
            True,

        "champion":
            "BASE_PLUS_REGIME",

        "version":
            version,

        "live_engine":
            engine_name,

        "feed_path":
            str(feed_path),

        "state_path":
            str(state_path)

    }


# ============================================================
# BUILD
# 赤いTracebackを出さず、安全停止する
# ============================================================

try:

    FORWARD_PAPER_BUILD = (
        _build_forward_paper_system()
    )

except Exception as e:

    FORWARD_PAPER_READY = False

    FORWARD_PAPER_BUILD = {

        "ready":
            False,

        "error_type":
            type(e).__name__,

        "message":
            str(e)

    }

    print(
        "=" * 90
    )

    print(
        "FORWARD PAPER BUILD STOPPED SAFELY"
    )

    print(
        "=" * 90
    )

    print(
        "Error type:",
        type(e).__name__
    )

    print(
        "Message:",
        str(e)
    )

    print()

    print(
        "No Champion/model/feature was modified."
    )

    print(
        "No paper/live order was generated."
    )


## 元セルindex 85
構文状態：valid


In [ ]:
# ============================================================
# FORWARD PAPER DATA INGESTION BRIDGE
# Frozen Champion 用
#
# 目的:
#   1. 新規15分足を安全に受け取る
#   2. データを厳格に検証
#   3. Forward Feed に atomic append
#   4. run_forward_paper() を実行
#   5. API接続前の受け口を完成させる
#
# IMPORTANT:
#   - Champion/model/features は変更しない
#   - 実注文は送らない
#   - 未確定15分足は拒否
#   - 重複バーは追加しない
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import os
import shutil
from datetime import datetime, timezone


# ============================================================
# 0. CONFIG
# ============================================================

PAIR = "USDJPY"
TIMEFRAME_MINUTES = 15

EXPECTED_COLS = [
    "timestamp",
    "open",
    "high",
    "low",
    "close",
]

# 現在FreezeされているChampionの既知cutoff。
# runtime contractから取得できない場合だけ使う。
KNOWN_FROZEN_LATEST = pd.Timestamp(
    "2026-09-01 00:00:00",
    tz="UTC",
)


# ============================================================
# 1. Champion directory discovery
# ============================================================

def find_frozen_champion_directory():

    root = Path("production_champion")

    if not root.exists():
        return None

    candidates = []

    for p in root.iterdir():

        if not p.is_dir():
            continue

        name = p.name.lower()

        if (
            "champion_v1_base_plus_regime" in name
            or "base_plus_regime" in name
        ):
            candidates.append(p)

    if len(candidates) == 0:
        return None

    # 名前順で最新を取る
    candidates = sorted(
        candidates,
        key=lambda x: x.name,
    )

    return candidates[-1]


CHAMPION_DIR = find_frozen_champion_directory()


# ============================================================
# 2. Runtime path discovery
# ============================================================

BRIDGE_READY = True

if CHAMPION_DIR is None:

    print("=" * 80)
    print("STOP")
    print("=" * 80)

    print(
        "Frozen Champion directory が見つかりません。"
    )

    BRIDGE_READY = False

else:

    RUNTIME_DIR = CHAMPION_DIR / "runtime"

    PAPER_DIR = (
        RUNTIME_DIR
        / "paper_forward"
    )

    PAPER_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    # 既存Forward Feed探索
    feed_candidates = list(
        PAPER_DIR.glob(
            "*forward_feed*.csv"
        )
    )

    if len(feed_candidates) > 0:

        FORWARD_FEED_PATH = sorted(
            feed_candidates
        )[0]

    else:

        FORWARD_FEED_PATH = (
            PAPER_DIR
            / "usdjpy_15m_forward_feed.csv"
        )

    INBOX_PATH = (
        PAPER_DIR
        / "incoming_usdjpy_15m.csv"
    )

    AUDIT_LOG_PATH = (
        PAPER_DIR
        / "forward_ingestion_audit.csv"
    )


# ============================================================
# 3. Runtime contractからFrozen Latestを探す
# ============================================================

def _find_timestamp_recursive(obj, preferred_keys):

    if isinstance(obj, dict):

        # 優先キー
        for key in preferred_keys:

            if key in obj:

                try:

                    ts = pd.to_datetime(
                        obj[key],
                        utc=True,
                        errors="coerce",
                    )

                    if not pd.isna(ts):
                        return ts

                except Exception:
                    pass

        # 再帰
        for value in obj.values():

            found = _find_timestamp_recursive(
                value,
                preferred_keys,
            )

            if found is not None:
                return found

    elif isinstance(obj, list):

        for value in obj:

            found = _find_timestamp_recursive(
                value,
                preferred_keys,
            )

            if found is not None:
                return found

    return None


def get_frozen_latest():

    if not BRIDGE_READY:
        return KNOWN_FROZEN_LATEST

    preferred_keys = [
        "canonical_latest",
        "frozen_latest",
        "frozen_history_latest",
        "latest_bar",
        "frozen_end",
    ]

    json_files = list(
        CHAMPION_DIR.rglob("*.json")
    )

    # runtime contractを優先
    json_files = sorted(
        json_files,
        key=lambda p:
        (
            "runtime_contract" not in p.name.lower(),
            str(p),
        )
    )

    for path in json_files:

        try:

            with open(
                path,
                "r",
                encoding="utf-8",
            ) as f:

                obj = json.load(f)

            found = _find_timestamp_recursive(
                obj,
                preferred_keys,
            )

            if found is not None:
                return found

        except Exception:
            pass

    return KNOWN_FROZEN_LATEST


FROZEN_LATEST = get_frozen_latest()


# ============================================================
# 4. DataFrame normalizer
# ============================================================

def normalize_15m_bars(df):

    if df is None:
        return None, "INPUT_IS_NONE"

    if not isinstance(df, pd.DataFrame):
        return None, "INPUT_IS_NOT_DATAFRAME"

    if len(df) == 0:
        return pd.DataFrame(
            columns=EXPECTED_COLS
        ), None

    x = df.copy()

    # ----------------------------------------
    # DatetimeIndex対応
    # ----------------------------------------

    if (
        isinstance(
            x.index,
            pd.DatetimeIndex,
        )
        and "timestamp" not in x.columns
    ):

        idx_name = (
            x.index.name
            if x.index.name is not None
            else "timestamp"
        )

        x = x.reset_index()

        if idx_name in x.columns:

            x = x.rename(
                columns={
                    idx_name: "timestamp"
                }
            )

        elif x.columns[0] != "timestamp":

            x = x.rename(
                columns={
                    x.columns[0]:
                    "timestamp"
                }
            )

    # ----------------------------------------
    # column alias対応
    # ----------------------------------------

    lower_map = {
        str(c).lower().strip(): c
        for c in x.columns
    }

    aliases = {

        "timestamp": [
            "timestamp",
            "time",
            "datetime",
            "date",
        ],

        "open": [
            "open",
            "o",
        ],

        "high": [
            "high",
            "h",
        ],

        "low": [
            "low",
            "l",
        ],

        "close": [
            "close",
            "c",
        ],
    }

    rename_map = {}

    for target, alias_list in aliases.items():

        if target in x.columns:
            continue

        for alias in alias_list:

            if alias in lower_map:

                rename_map[
                    lower_map[alias]
                ] = target

                break

    if len(rename_map) > 0:

        x = x.rename(
            columns=rename_map
        )

    # ----------------------------------------
    # 必須column確認
    # ----------------------------------------

    missing = [
        c
        for c in EXPECTED_COLS
        if c not in x.columns
    ]

    if len(missing) > 0:

        return (
            None,
            "MISSING_COLUMNS: "
            + str(missing),
        )

    x = x[
        EXPECTED_COLS
    ].copy()

    # ----------------------------------------
    # timestamp
    # ----------------------------------------

    x["timestamp"] = pd.to_datetime(
        x["timestamp"],
        utc=True,
        errors="coerce",
    )

    # ----------------------------------------
    # OHLC
    # ----------------------------------------

    for col in [
        "open",
        "high",
        "low",
        "close",
    ]:

        x[col] = pd.to_numeric(
            x[col],
            errors="coerce",
        )

    # ----------------------------------------
    # NaN
    # ----------------------------------------

    if x[
        EXPECTED_COLS
    ].isna().any().any():

        return (
            None,
            "NAN_OR_INVALID_VALUES",
        )

    x = (
        x
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    # ----------------------------------------
    # timestamp duplicate
    # ----------------------------------------

    if x["timestamp"].duplicated().any():

        return (
            None,
            "DUPLICATE_TIMESTAMPS",
        )

    # ----------------------------------------
    # exact 15m grid
    # ----------------------------------------

    ts = x["timestamp"]

    grid_ok = (
        (ts.dt.minute % TIMEFRAME_MINUTES == 0)
        & (ts.dt.second == 0)
        & (ts.dt.microsecond == 0)
    )

    if not bool(grid_ok.all()):

        bad = x.loc[
            ~grid_ok,
            "timestamp",
        ]

        return (
            None,
            "OFF_GRID_TIMESTAMP: "
            + str(
                bad.head().tolist()
            ),
        )

    # ----------------------------------------
    # positive prices
    # ----------------------------------------

    positive = (
        x[
            [
                "open",
                "high",
                "low",
                "close",
            ]
        ]
        > 0
    ).all(axis=1)

    if not bool(positive.all()):

        return (
            None,
            "NON_POSITIVE_PRICE",
        )

    # ----------------------------------------
    # OHLC logical integrity
    # ----------------------------------------

    high_ok = (
        x["high"]
        >= x[
            [
                "open",
                "close",
                "low",
            ]
        ].max(axis=1)
    )

    low_ok = (
        x["low"]
        <= x[
            [
                "open",
                "close",
                "high",
            ]
        ].min(axis=1)
    )

    if not bool(
        (
            high_ok
            & low_ok
        ).all()
    ):

        return (
            None,
            "INVALID_OHLC_RELATIONSHIP",
        )

    # ----------------------------------------
    # closed bar only
    # ----------------------------------------

    now_utc = pd.Timestamp.now(
        tz="UTC"
    )

    close_times = (
        x["timestamp"]
        + pd.Timedelta(
            minutes=TIMEFRAME_MINUTES
        )
    )

    closed_ok = (
        close_times
        <= now_utc
    )

    if not bool(closed_ok.all()):

        bad = x.loc[
            ~closed_ok,
            "timestamp",
        ]

        return (
            None,
            "UNFINISHED_BAR: "
            + str(
                bad.head().tolist()
            ),
        )

    return x, None


# ============================================================
# 5. Read existing feed safely
# ============================================================

def read_existing_forward_feed():

    if not BRIDGE_READY:

        return pd.DataFrame(
            columns=EXPECTED_COLS
        )

    if not FORWARD_FEED_PATH.exists():

        return pd.DataFrame(
            columns=EXPECTED_COLS
        )

    try:

        if FORWARD_FEED_PATH.stat().st_size == 0:

            return pd.DataFrame(
                columns=EXPECTED_COLS
            )

        df = pd.read_csv(
            FORWARD_FEED_PATH
        )

        normalized, error = (
            normalize_15m_bars(df)
        )

        if error is not None:

            print(
                "[WARNING] Existing feed "
                "could not be normalized:"
            )

            print(error)

            return pd.DataFrame(
                columns=EXPECTED_COLS
            )

        return normalized

    except Exception as e:

        print(
            "[WARNING] Existing feed "
            "read failed:"
        )

        print(
            type(e).__name__,
            str(e),
        )

        return pd.DataFrame(
            columns=EXPECTED_COLS
        )


# ============================================================
# 6. Atomic write
# ============================================================

def atomic_write_csv(
    df,
    path,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = path.with_name(
        path.name + ".tmp"
    )

    df.to_csv(
        temp_path,
        index=False,
    )

    os.replace(
        temp_path,
        path,
    )


# ============================================================
# 7. Audit logger
# ============================================================

def append_audit_log(
    status,
    rows_received,
    rows_added,
    latest_timestamp=None,
    message="",
):

    if not BRIDGE_READY:
        return

    record = pd.DataFrame(
        [
            {
                "event_time_utc":
                    pd.Timestamp.now(
                        tz="UTC"
                    ),

                "status":
                    status,

                "rows_received":
                    int(rows_received),

                "rows_added":
                    int(rows_added),

                "latest_timestamp":
                    latest_timestamp,

                "message":
                    str(message),
            }
        ]
    )

    if AUDIT_LOG_PATH.exists():

        try:

            old = pd.read_csv(
                AUDIT_LOG_PATH
            )

            record = pd.concat(
                [
                    old,
                    record,
                ],
                ignore_index=True,
            )

        except Exception:
            pass

    atomic_write_csv(
        record,
        AUDIT_LOG_PATH,
    )


# ============================================================
# 8. Submit new closed bars
# ============================================================

def submit_forward_bars(
    new_df
):

    if not BRIDGE_READY:

        return {
            "passed": False,
            "reason":
                "BRIDGE_NOT_READY",
            "rows_added": 0,
        }

    normalized, error = (
        normalize_15m_bars(
            new_df
        )
    )

    if error is not None:

        print("=" * 80)
        print("FORWARD BAR REJECTED")
        print("=" * 80)

        print(error)

        append_audit_log(
            status="REJECTED",
            rows_received=(
                0
                if new_df is None
                else len(new_df)
            ),
            rows_added=0,
            message=error,
        )

        return {
            "passed": False,
            "reason": error,
            "rows_added": 0,
        }

    if len(normalized) == 0:

        return {
            "passed": True,
            "reason":
                "NO_ROWS",
            "rows_added": 0,
        }

    existing = (
        read_existing_forward_feed()
    )

    # ----------------------------------------
    # Frozen cutoff
    # ----------------------------------------

    cutoff = FROZEN_LATEST

    if len(existing) > 0:

        existing_latest = (
            existing[
                "timestamp"
            ].max()
        )

        cutoff = max(
            cutoff,
            existing_latest,
        )

    # ----------------------------------------
    # Only strictly new bars
    # ----------------------------------------

    candidates = (
        normalized[
            normalized[
                "timestamp"
            ] > cutoff
        ]
        .copy()
    )

    if len(candidates) == 0:

        print("=" * 80)
        print("NO NEW FORWARD BARS")
        print("=" * 80)

        print(
            "Latest accepted cutoff:",
            cutoff,
        )

        append_audit_log(
            status="NO_NEW_ROWS",
            rows_received=len(
                normalized
            ),
            rows_added=0,
            latest_timestamp=cutoff,
            message=(
                "All rows already "
                "processed or frozen."
            ),
        )

        return {
            "passed": True,
            "reason":
                "NO_NEW_ROWS",
            "rows_added": 0,
        }

    # ----------------------------------------
    # Existing feed duplicates
    # ----------------------------------------

    if len(existing) > 0:

        old_times = set(
            existing[
                "timestamp"
            ].tolist()
        )

        candidates = (
            candidates[
                ~candidates[
                    "timestamp"
                ].isin(old_times)
            ]
            .copy()
        )

    if len(candidates) == 0:

        return {
            "passed": True,
            "reason":
                "DUPLICATE_ONLY",
            "rows_added": 0,
        }

    # ----------------------------------------
    # backup
    # ----------------------------------------

    if (
        FORWARD_FEED_PATH.exists()
        and FORWARD_FEED_PATH.stat().st_size > 0
    ):

        stamp = datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%d_%H%M%S"
        )

        backup_path = (
            PAPER_DIR
            / (
                "feed_backup_"
                + stamp
                + ".csv"
            )
        )

        try:

            shutil.copy2(
                FORWARD_FEED_PATH,
                backup_path,
            )

        except Exception:
            pass

    # ----------------------------------------
    # combine
    # ----------------------------------------

    combined = pd.concat(
        [
            existing,
            candidates,
        ],
        ignore_index=True,
    )

    combined = (
        combined
        .sort_values("timestamp")
        .drop_duplicates(
            subset=["timestamp"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    atomic_write_csv(
        combined,
        FORWARD_FEED_PATH,
    )

    latest_added = (
        candidates[
            "timestamp"
        ].max()
    )

    append_audit_log(
        status="APPENDED",
        rows_received=len(
            normalized
        ),
        rows_added=len(
            candidates
        ),
        latest_timestamp=latest_added,
        message="Validated closed 15m bars appended.",
    )

    print("=" * 80)
    print("FORWARD BAR INGESTION PASSED")
    print("=" * 80)

    print(
        "Rows received:",
        len(normalized),
    )

    print(
        "Rows added:",
        len(candidates),
    )

    print(
        "First added:",
        candidates[
            "timestamp"
        ].min(),
    )

    print(
        "Last added:",
        latest_added,
    )

    print(
        "Forward feed:",
        FORWARD_FEED_PATH,
    )

    return {
        "passed": True,
        "reason":
            "APPENDED",
        "rows_added":
            len(candidates),
        "latest_timestamp":
            latest_added,
    }


# ============================================================
# 9. Execute paper engine once
# ============================================================

def run_forward_paper_once(
    new_df=None
):

    print()
    print("=" * 80)
    print("FORWARD PAPER ONE-SHOT")
    print("=" * 80)

    # ----------------------------------------
    # optional ingestion
    # ----------------------------------------

    if new_df is not None:

        result = submit_forward_bars(
            new_df
        )

        if not result.get(
            "passed",
            False,
        ):

            print()
            print(
                "Paper Engine NOT run "
                "because ingestion failed."
            )

            return result

    # ----------------------------------------
    # engine existence
    # ----------------------------------------

    fn = globals().get(
        "run_forward_paper"
    )

    if not callable(fn):

        print(
            "run_forward_paper() "
            "was not found."
        )

        return {
            "passed": False,
            "reason":
                "RUN_FORWARD_PAPER_NOT_FOUND",
        }

    # ----------------------------------------
    # paper engine
    # ----------------------------------------

    try:

        output = fn()

        print()
        print("=" * 80)
        print(
            "FORWARD PAPER ENGINE "
            "CALL COMPLETE"
        )
        print("=" * 80)

        return {
            "passed": True,
            "reason":
                "ENGINE_EXECUTED",
            "output":
                output,
        }

    except Exception as e:

        print()
        print("=" * 80)
        print(
            "PAPER ENGINE SAFE STOP"
        )
        print("=" * 80)

        print(
            type(e).__name__,
            str(e),
        )

        append_audit_log(
            status=
                "ENGINE_SAFE_STOP",
            rows_received=0,
            rows_added=0,
            message=(
                type(e).__name__
                + ": "
                + str(e)
            ),
        )

        return {
            "passed": False,
            "reason":
                "ENGINE_EXCEPTION",
            "exception":
                str(e),
        }


# ============================================================
# 10. Incoming CSV loader
# ============================================================

def load_incoming_csv():

    if not BRIDGE_READY:
        return None

    if not INBOX_PATH.exists():

        # template only
        template = pd.DataFrame(
            columns=EXPECTED_COLS
        )

        template.to_csv(
            INBOX_PATH,
            index=False,
        )

        return None

    try:

        if INBOX_PATH.stat().st_size == 0:
            return None

        df = pd.read_csv(
            INBOX_PATH
        )

        if len(df) == 0:
            return None

        return df

    except Exception as e:

        print(
            "Incoming CSV read failed:",
            type(e).__name__,
            str(e),
        )

        return None


# ============================================================
# 11. System health report
# ============================================================

print()
print("=" * 80)
print("FORWARD PAPER INGESTION BRIDGE")
print("=" * 80)

print(
    "Bridge ready:",
    BRIDGE_READY,
)

if BRIDGE_READY:

    print(
        "Champion:",
        CHAMPION_DIR.name,
    )

    print(
        "Frozen latest:",
        FROZEN_LATEST,
    )

    print(
        "Paper directory:",
        PAPER_DIR,
    )

    print(
        "Forward feed:",
        FORWARD_FEED_PATH,
    )

    print(
        "Incoming file:",
        INBOX_PATH,
    )

    print(
        "Audit log:",
        AUDIT_LOG_PATH,
    )

    print(
        "run_forward_paper exists:",
        callable(
            globals().get(
                "run_forward_paper"
            )
        ),
    )


# ============================================================
# 12. Optional source detection
# ============================================================

AUTO_SOURCE = None
AUTO_SOURCE_NAME = None

for name in [
    "NEW_FORWARD_BARS",
    "new_forward_bars",
    "new_bars",
]:

    obj = globals().get(name)

    if isinstance(
        obj,
        pd.DataFrame,
    ):

        AUTO_SOURCE = obj.copy()
        AUTO_SOURCE_NAME = name
        break


if AUTO_SOURCE is None:

    inbox_df = load_incoming_csv()

    if isinstance(
        inbox_df,
        pd.DataFrame,
    ):

        AUTO_SOURCE = inbox_df
        AUTO_SOURCE_NAME = (
            str(INBOX_PATH)
        )


# ============================================================
# 13. Final readiness decision
# ============================================================

print()
print("=" * 80)
print("FINAL BRIDGE CHECK")
print("=" * 80)

checks = {

    "champion_directory":
        CHAMPION_DIR is not None,

    "paper_directory":
        (
            BRIDGE_READY
            and PAPER_DIR.exists()
        ),

    "frozen_cutoff":
        (
            FROZEN_LATEST
            is not None
        ),

    "paper_engine":
        callable(
            globals().get(
                "run_forward_paper"
            )
        ),

    "paper_only":
        True,

    "real_orders":
        False,
}

for k, v in checks.items():

    print(
        f"{k}: {v}"
    )


BRIDGE_FINAL_PASS = (
    all(checks.values())
)

print()
print(
    "FORWARD_INGESTION_READY:",
    BRIDGE_FINAL_PASS,
)


# ============================================================
# 14. Do NOT invent market data
# ============================================================

if AUTO_SOURCE is not None:

    print()
    print(
        "Detected incoming source:",
        AUTO_SOURCE_NAME,
    )

    print(
        "Rows:",
        len(AUTO_SOURCE),
    )

    # 新しい実データがある場合のみ処理
    FORWARD_BRIDGE_RESULT = (
        run_forward_paper_once(
            AUTO_SOURCE
        )
    )

else:

    print()
    print("=" * 80)
    print("READY — WAITING FOR NEW 15M BARS")
    print("=" * 80)

    print(
        "No market bars were invented "
        "or generated."
    )

    print()
    print(
        "Incoming CSV template:"
    )

    print(
        INBOX_PATH
    )

    print()
    print(
        "Required columns:"
    )

    print(
        EXPECTED_COLS
    )

    print()
    print(
        "When real closed 15m bars "
        "are supplied, run:"
    )

    print(
        "run_forward_paper_once("
        "your_dataframe)"
    )

    FORWARD_BRIDGE_RESULT = {
        "passed":
            BRIDGE_FINAL_PASS,

        "reason":
            "WAITING_FOR_MARKET_DATA",
    }


print()
print("=" * 80)
print("CELL COMPLETE")
print("=" * 80)

print(
    "Bridge passed:",
    BRIDGE_FINAL_PASS,
)

print(
    "Result:",
    FORWARD_BRIDGE_RESULT,
)


## 元セルindex 86
構文状態：valid


In [ ]:
# ============================================================
# FORWARD BRIDGE FINAL CHECK FIX
# real_orders=False は Paper Trading では正常
# ============================================================

print()
print("=" * 80)
print("FORWARD PAPER BRIDGE - FINAL CHECK FIX")
print("=" * 80)

# 前セルの checks を利用
if "checks" not in globals():
    raise RuntimeError(
        "Previous Forward Paper Ingestion Bridge cell has not been executed."
    )

# real_orders=False を「実注文が無効」という正常条件に変換
FINAL_BRIDGE_CHECKS = {
    "champion_directory": bool(checks.get("champion_directory", False)),
    "paper_directory": bool(checks.get("paper_directory", False)),
    "frozen_cutoff": bool(checks.get("frozen_cutoff", False)),
    "paper_engine": bool(checks.get("paper_engine", False)),
    "paper_only": bool(checks.get("paper_only", False)),
    "real_orders_disabled": checks.get("real_orders", None) is False,
}

for name, passed in FINAL_BRIDGE_CHECKS.items():
    print(f"{name}: {passed}")

BRIDGE_FINAL_PASS = all(FINAL_BRIDGE_CHECKS.values())

print()
print("=" * 80)
print("FINAL RESULT")
print("=" * 80)

print("FORWARD_INGESTION_READY:", BRIDGE_FINAL_PASS)

if BRIDGE_FINAL_PASS:

    FORWARD_BRIDGE_RESULT = {
        "passed": True,
        "reason": "WAITING_FOR_MARKET_DATA",
    }

    print()
    print("STATUS: PASS")
    print("Forward Paper Trading infrastructure is ready.")
    print("Real orders are DISABLED as required.")
    print("System is safely waiting for new closed USDJPY 15m bars.")

    print()
    print("Result:", FORWARD_BRIDGE_RESULT)

else:

    failed = [
        name
        for name, passed in FINAL_BRIDGE_CHECKS.items()
        if not passed
    ]

    FORWARD_BRIDGE_RESULT = {
        "passed": False,
        "reason": "BRIDGE_CHECK_FAILED",
        "failed_checks": failed,
    }

    print()
    print("STATUS: STOP")
    print("Failed checks:", failed)
    print("Result:", FORWARD_BRIDGE_RESULT)

print()
print("=" * 80)
print("CELL COMPLETE")
print("=" * 80)
